In [5]:
import cv2
import mediapipe as mp
import time

In [6]:
class FaceMeshDetector:
    def __init__(self, staticMode=False, maxFaces=2, minDetectionCon=0.5, minTrackCon=0.5):
        self.staticMode = staticMode
        self.maxFaces = maxFaces
        self.minDetectionCon = minDetectionCon
        self.minTrackCon = minTrackCon

        self.mpDraw = mp.solutions.drawing_utils
        self.mpFaceMesh = mp.solutions.face_mesh
        self.faceMesh = self.mpFaceMesh.FaceMesh(
            # num_faces = self.maxFaces, min_face_detection_confidence = self.minDetectionCon, min_tracking_confidence = self.minTrackCon
        )
        self.drawSpec = self.mpDraw.DrawingSpec(thickness=1, circle_radius=2)

    def findFaceMesh(self, img, draw=True):
        self.imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        self.results = self.faceMesh.process(self.imgRGB)
        faces = []

        if self.results.multi_face_landmarks:
            for faceLms in self.results.multi_face_landmarks:
                if draw:
                    self.mpDraw.draw_landmarks(
                        image=img, landmark_list=faceLms,
                        landmark_drawing_spec=self.drawSpec
                    )
                face = []
                for id, lm in enumerate(faceLms.landmark):
                    # Get the dimensions of the image
                    ih, iw, ic = img.shape
                    x, y = int(lm.x * iw), int(lm.y * ih)
                    # Append each landmark's (x, y) coordinates to the face list
                    face.append([x, y])
                faces.append(face)
        return img, faces


In [7]:
def main():
    cap = cv2.VideoCapture(0)
    pTime = 0
    detector = FaceMeshDetector(maxFaces=2)

    while True:
        success, img = cap.read()
        if not success:
            break

        img, faces = detector.findFaceMesh(img)
        if len(faces) != 0:
            print(faces[0])

        cTime = time.time()
        fps = 1 / (cTime - pTime)
        pTime = cTime

        cv2.putText(
            img, f'FPS: {int(fps)}', (20, 70),
            cv2.FONT_HERSHEY_PLAIN, 3, (0, 255, 0), 3
        )
        cv2.imshow("Image", img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [ ]:

if __name__ == "__main__":
    main()